In [3]:
import time
import numpy as np
from scipy.spatial import cKDTree  

class LargeDetector:
    def __init__(self, R=15.0, N=2000, r=0.1, a=0.5):
        self.R = R
        self.N = N
        self.r = r
        self.a = a
        
        self.pmt_positions = self.generate_pmt_positions()
        self.pmt_angular_radius = np.arcsin(self.r / self.R)
        
        # создаем KD-дерево по нормализованным позициям ФЭУ на единичной сфере для быстрого поиска
        norm_pmts = self.pmt_positions / self.R
        self.kd_tree = cKDTree(norm_pmts)

    # равномерное распределение N точек на сфере
    def generate_pmt_positions(self):
        positions = []
        phi = np.pi * (np.sqrt(5.0) - 1.0)
        for i in range(self.N):
            y = 1.0 - (i / float(self.N - 1)) * 2.0
            radius_at_y = np.sqrt(1.0 - y * y)
            theta = phi * i
            
            x = np.cos(theta) * radius_at_y
            z = np.sin(theta) * radius_at_y
            
            pos = np.array([x, y, z]) * self.R
            positions.append(pos)
        return np.array(positions)

    def check_hits_optimized(self, hit_points):
        n_photons = len(hit_points)
        norm_hits = hit_points / np.linalg.norm(hit_points, axis=1, keepdims=True)
        
        pmt_hits = np.zeros(self.N, dtype=int)
        
        distances, indices = self.kd_tree.query(norm_hits, k=1)
        
        norm_pmts = self.pmt_positions / self.R
        for i, hit_idx in enumerate(indices):
            cos_angle = np.dot(norm_hits[i], norm_pmts[hit_idx])
            cos_angle = np.clip(cos_angle, -1.0, 1.0)
            angle = np.arccos(cos_angle)
            
            if angle <= self.pmt_angular_radius:
                pmt_hits[hit_idx] += 1
                
        return pmt_hits


class SmartSource:
    def __init__(self, Y=10000):
        self.Y = Y

    def generate_photons_generator(self, energy, pos=(0.0, 0.0, 0.0), batch_size=500):
        mean_photons = energy * self.Y
        n_photons = np.random.poisson(mean_photons)
        
        phi = np.random.uniform(0, 2 * np.pi, n_photons)
        costheta = np.random.uniform(-1.0, 1.0, n_photons)
        sintheta = np.sqrt(1.0 - costheta**2)
        
        directions = np.column_stack((sintheta * np.cos(phi), sintheta * np.sin(phi), costheta))
        positions = np.tile(pos, (n_photons, 1))
        
        for i in range(0, n_photons, batch_size):
            end_idx = min(i + batch_size, n_photons)
            yield positions[i:end_idx], directions[i:end_idx]

detector = LargeDetector(R=15.0, N=2000, r=0.1, a=0.5)
source = SmartSource(Y=10000)

energy = 1.0  # МэВ
n_events = 100

start_time = time.time()

total_charges = []
for event in range(n_events):
    pmt_hits = np.zeros(detector.N, dtype=int)
    for r0, directions in source.generate_photons_generator(energy, pos=(0.0, 0.0, 0.0), batch_size=500):
        b = 2 * np.sum(r0 * directions, axis=1)
        c = np.sum(r0**2, axis=1) - detector.R**2
        t = (-b + np.sqrt(b**2 - 4 * c)) / 2.0
        
        hit_points = r0 + directions * t[:, np.newaxis]
        pmt_hits += detector.check_hits_optimized(hit_points)
        
    pe_per_pmt = np.random.poisson(detector.a * pmt_hits)
    total_charges.append(np.sum(pe_per_pmt))

elapsed = time.time() - start_time
mean_charge = np.mean(total_charges)

print(f"Успешно смоделировано событий: {n_events}")
print(f"Средний собранный заряд в большом детекторе: {mean_charge:.1f} ф.э.")
print(f"Время выполнения: {elapsed:.2f} сек.")

Успешно смоделировано событий: 100
Средний собранный заряд в большом детекторе: 109.0 ф.э.
Время выполнения: 18.98 сек.
